# STAR — Kaggle V5: SO SANH best.pth (cua ban) vs X-VLM GOC tren old-test THAT

**Khong dung distractor** — gallery = dung **1.978 anh that** (protocol giong paper). Moi anh vua la
**query** vua la **ung vien gallery**; GT cua moi query = chinh anh do.

**2 model x 3 tang:**
- `trained` = best.pth ban vua train (auto-chon best_metric cao nhat). Neu train POSE-ON → eval CO pose:
  notebook **tu chay YOLOv8-pose** sinh keypoints cho 1.978 anh (thay ViTPose — mmpose dung chung voi env pin se vo).
- `base`    = X-VLM 16M GOC (chua finetune, **luon pose-OFF**) — de biet finetune GIUP hay HAI transfer that.
- 3 tang: **stage1** (cosine, KHONG cross) → **rerank** (+ITM) → **gale_shapley** (+1-1).

Pose fuse vao dac trung ANH (ca 1.978 anh o gallery), khong phai text. Cross-encoder KHONG dung pose → rerank cong bang ca 2 model.

**Add Input:** 1) `star-oldtest` (attr.json + test/) · 2) `star-10k-hard` (xvlm_16m_base.th, hoac gdown) ·
3) dataset chua **best.pth** · 4) *(tuy chon)* dataset chua `*vitpose*.json` cho old-test (neu co thi dung luon, khoi chay YOLO).

**Thoi gian:** setup ~10' + YOLO-pose ~2' + ~8-12'/(model×mode) → ~50-65'.

In [ ]:
# [1/8] GPU + clone/pull repo (can ban MOI: inference + pose-in-pipeline + extract_pose_yolo)
import os, pathlib, subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert gpu, "KHONG THAY GPU — bat Settings > Accelerator = GPU T4!"
print("GPU:", gpu)
os.chdir("/kaggle/working")
if pathlib.Path("star/.git").exists():
    !cd star && git pull -q
else:
    !git clone -q https://github.com/Khanhhh239/Model_XVLM_Training.git star
os.chdir("/kaggle/working/star")
assert "keypoints" in pathlib.Path("src/star/inference/pipeline.py").read_text(), "repo cu — pull lai!"
assert pathlib.Path("scripts/extract_pose_yolo.py").exists(), "repo cu — thieu extract_pose_yolo, pull lai!"
print("repo OK (inference + pose + yolo):", os.getcwd())

In [ ]:
# [2/8] Env pinned + X-VLM + 4 patches
!python scripts/kaggle_setup.py

In [ ]:
# [3/8] Tim inputs: old-test, X-VLM base, best.pth (best_metric cao nhat), va (tuy chon) ViTPose json
import glob, pathlib, torch

def find_all(p):
    return sorted(set(glob.glob(f"/kaggle/input/*/{p}") + glob.glob(f"/kaggle/input/*/**/{p}", recursive=True)))

attr = find_all("attr.json")
assert attr, "Khong thay attr.json — Add Input dataset star-oldtest!"
ATTR = attr[0]; OLD_ROOT = str(pathlib.Path(ATTR).parent)
probe = find_all("test/0.jpg") or find_all("0.jpg")
assert probe, "Khong thay anh test/0.jpg trong dataset old-test"
if not pathlib.Path(OLD_ROOT, "test/0.jpg").exists():
    OLD_ROOT = str(pathlib.Path(probe[0]).parents[1])

base = find_all("xvlm_16m_base.th")
if base:
    BASE_CKPT = base[0]
else:
    os.makedirs("data/checkpoints", exist_ok=True)
    !gdown -q 1iXgITaSbQ1oGPPvGaV0Hlae4QiJG5gx0 -O data/checkpoints/xvlm_16m_base.th
    BASE_CKPT = "data/checkpoints/xvlm_16m_base.th"
assert pathlib.Path(BASE_CKPT).stat().st_size > 8e8

cands = find_all("best.pth")
assert cands, "Khong thay best.pth nao — Add Input dataset chua best.pth ban da train!"
print("Tim thay best.pth:")
scored = []
for c in cands:
    try:
        raw = torch.load(c, map_location="cpu", weights_only=False)
        bm = raw.get("best_metric")
        pe = ((raw.get("extra") or {}).get("cfg") or {}).get("model", {}).get("pose_enabled")
        del raw
    except Exception as e:
        bm, pe = None, f"(loi:{e})"
    scored.append((bm if bm is not None else -1, c, pe))
    print(f"  {c} | best_metric={bm} pose_enabled={pe}")
scored.sort(reverse=True)
_, BEST_CKPT, BEST_POSE = scored[0]
POSE_ON = str(BEST_POSE).strip().lower() == "true"      # ckpt nay co train voi pose khong
VITPOSE = (find_all("*vitpose*.json") or [None])[0]
print(f"\n=> CHON: {BEST_CKPT} (best_metric={scored[0][0]}, pose_on={POSE_ON})")
print("OLD_ROOT =", OLD_ROOT, "| BASE_CKPT =", BASE_CKPT, "| VITPOSE =", VITPOSE or "(chua co)")

In [ ]:
# [4/8] (pose) Neu ckpt POSE-ON va chua co ViTPose json -> tu chay YOLOv8-pose tren 1.978 anh
RUN_YOLO_POSE = True    # False neu muon ep eval POSE-OFF du ckpt pose-on
if POSE_ON and VITPOSE is None and RUN_YOLO_POSE:
    print("ckpt POSE-ON & chua co json -> cai ultralytics + chay YOLOv8-pose (weight ~130MB, can Internet ON)...")
    !pip install -q ultralytics
    !python scripts/extract_pose_yolo.py --attr "{ATTR}" --image-root "{OLD_ROOT}" --out /kaggle/working/oldtest_yolo_pose.json --device 0
    if pathlib.Path("/kaggle/working/oldtest_yolo_pose.json").exists():
        VITPOSE = "/kaggle/working/oldtest_yolo_pose.json"
        print("=> VITPOSE =", VITPOSE)
    else:
        print("⚠️  YOLO-pose that bai -> se eval POSE-OFF")
elif VITPOSE:
    print("Dung ViTPose json co san:", VITPOSE)
else:
    print("Eval POSE-OFF (ckpt khong pose-on hoac RUN_YOLO_POSE=False)")

In [ ]:
# [5/8] Build manifest gallery-1978 (KHONG distractor) + keypoints (neu co pose json)
import json
import pandas as pd

MODES = ["human", "vlm"]   # human=source_caption (ngan) | vlm=caption chi tiet (giong train)
rows = [json.loads(l) for l in open(ATTR, encoding="utf-8")]
assert len(rows) == 1978, f"attr.json co {len(rows)} dong (mong doi 1978)"

pose_items = json.load(open(VITPOSE, encoding="utf-8")).get("items", {}) if VITPOSE else {}
def kpts_of(iid):
    it = pose_items.get(str(iid))
    if not it or it.get("status") != "ok" or not it.get("instances"):
        return [0.0] * 51 if VITPOSE else None     # zero-fill khi co json nhung anh nay thieu kpts
    W, H = it.get("width", 384), it.get("height", 384)
    flat = []
    for x, y, c in it["instances"][0]["keypoints_xyc"]:
        flat += [x / W, y / H, c]
    return flat if len(flat) == 51 else ([0.0] * 51 if VITPOSE else None)

def cap_of(r, mode):
    return (r.get("source_caption") or r["caption"])[0] if mode == "human" else r["caption"][0]

MANIFESTS = {}
for mode in MODES:
    out = [dict(image_path=r["image"], caption=cap_of(r, mode), split="valb",
                sequence_id=f'o{r["image_id"]}', scene=f'o{r["image_id"]}', action="q",
                image_id=str(r["image_id"]), bbox=None, keypoints=kpts_of(r["image_id"])) for r in rows]
    df = pd.DataFrame(out)
    assert df.image_id.nunique() == 1978 and (df.caption != "").sum() == 1978
    mp = f"/kaggle/working/oldtest_{mode}.parquet"; df.to_parquet(mp, index=False); MANIFESTS[mode] = mp
KCOV = pd.read_parquet(MANIFESTS["human"]).keypoints.notna().mean() if VITPOSE else 0.0
print(f"manifests {MODES} | 1978 query=gallery (khong distractor) | kpts coverage: {KCOV:.0%}")

In [ ]:
# [6/8] Helper: dung model (trained / base) + chay pipeline 3 tang
import sys, torch
sys.path.insert(0, "src")
from star.config import load_config, _merge
from star.data import PABDataset
from star.inference import run_pipeline
from star.models import STARModel

def build_trained():
    raw = torch.load(BEST_CKPT, map_location="cpu", weights_only=False)
    cfg = load_config("configs/star_v3_10k_kaggle.yaml")
    emb = (raw.get("extra") or {}).get("cfg") or {}
    if "model" in emb:
        _merge(cfg.model, emb["model"])          # khoi phuc DUNG kien truc da train
    cfg.model.checkpoint = BASE_CKPT
    model = STARModel(cfg).to("cuda").eval()
    msg = model.load_state_dict(raw["model"], strict=False)
    active = (model.pose is not None) and bool(VITPOSE)
    print(f"  trained: missing={len(msg.missing_keys)} unexpected={len(msg.unexpected_keys)} "
          f"| POSE {'ON (keypoints)' if active else 'OFF'}")
    if model.pose is not None and not VITPOSE:
        print("  ⚠️  ckpt POSE-ON nhung khong co keypoints -> stage1 lech train/eval (rerank van cong bang)")
    del raw
    return model, cfg

def build_base():
    cfg = load_config("configs/star_v3_10k_kaggle.yaml")
    cfg.model.checkpoint = BASE_CKPT
    cfg.model.lora_enabled = False     # X-VLM GOC
    cfg.model.pose_enabled = False     # luon pose-OFF
    model = STARModel(cfg).to("cuda").eval()
    print("  base: X-VLM goc (khong LoRA, khong pose)")
    return model, cfg
print("helper san sang")

In [ ]:
# [7/8] CHAY SO SANH + BANG (trained vs base, delta)
import json, pathlib, time, torch
import pandas as pd
RESULTS = []
for name, builder in [("trained", build_trained), ("base_xvlm", build_base)]:
    print("=" * 88); print(">>>", name)
    model, cfg = builder()
    for mode in MODES:
        ds = PABDataset(MANIFESTS[mode], OLD_ROOT, model.backbone.tokenizer, split="valb",
                        image_size=cfg.data.image_size, max_token=cfg.data.max_token, train=False)
        t0 = time.time()
        res = run_pipeline(model, ds, "cuda", topk=100, batch_size=64, num_workers=2)
        for stage in ("stage1", "rerank", "gale_shapley"):
            RESULTS.append(dict(model=name, mode=mode, stage=stage,
                                **{k: round(v, 4) for k, v in res[stage].items()}))
        out = pathlib.Path(f"/kaggle/working/eval_v5/{name}_{mode}"); out.mkdir(parents=True, exist_ok=True)
        (out / "metrics.json").write_text(json.dumps(
            {s: res[s] for s in ("stage1", "rerank", "gale_shapley")} |
            {"gallery": res["gallery_size"], "queries": res["num_queries"]}, indent=2))
        print(f"  [{mode:5s}] stage1={res['stage1']['mAP']:.4f} rerank={res['rerank']['mAP']:.4f} "
              f"GS={res['gale_shapley']['mAP']:.4f}  ({(time.time()-t0)/60:.1f}')")
    del model; torch.cuda.empty_cache()

tab = pd.DataFrame(RESULTS)
print("\n" + tab.to_string(index=False))
print("\n========= mAP: trained vs base (delta = trained - base) =========")
mp = tab.pivot_table(index=["mode", "stage"], columns="model", values="mAP")
mp["delta"] = mp["trained"] - mp["base_xvlm"]
print(mp.to_string())
pathlib.Path("/kaggle/working/eval_v5/summary.json").write_text(json.dumps(RESULTS, indent=2))

## [8/8] Doc ket qua

- **`delta`** (trained − base): `>0` finetune GIUP transfer that (day manh v3c); `<0` finetune PHA bieu dien anh-that (overfit synthetic → doi chien luoc).
- **stage1 → rerank**: dong gop cross-encoder (khong dung pose → cong bang ca 2 model). Don manh nhat.
- **`human` vs `vlm`**: `vlm` cao hon → caption ngan (de nam nay) kho → can augment.
- **Pose**: trained eval CO pose (YOLOv8-pose ~ ViTPose, lech nhe ve detector). base luon pose-OFF.
- **Nhac luat**: test (masked) nam nay KHONG dung lam distractor/validation duoi moi hinh thuc.